In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [ ]:
from dotenv import load_dotenv, find_dotenv
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
from llama_index.embeddings.openai import OpenAIEmbedding

# Load environment variables
load_dotenv(find_dotenv())  
embed_model = OpenAIEmbedding(model_name = os.environ.get("EMBED_MODEL"))

if embed_model is None:
    raise ValueError("EMBED_MODEL environment variable is not set!")

# Load documents
documents = SimpleDirectoryReader(input_files=["../data/Pmg_lds.md"]).load_data()



In [ ]:
# merge into a single large document rather than the one document per page

from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[2048, 512, 128])

In [ ]:
nodes = node_parser.get_nodes_from_documents([document])

In [ ]:
len(nodes)

In [ ]:
nodes[30].text

In [ ]:
from llama_index.core.node_parser import get_leaf_nodes

leaf_nodes = get_leaf_nodes(nodes)
print(leaf_nodes[30].text)

In [ ]:
nodes_by_id = {node.node_id: node for node in nodes}

parent_node = nodes_by_id[leaf_nodes[30].parent_node.node_id]
print(parent_node.text)

In [ ]:
from llama_index.llms.openai import OpenAI

# llm = OpenAI(model ="gpt-3.5-turbo", temperature=0.1)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model=embed_model
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = node_parser

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import load_index_from_storage

if not os.path.exists("../VectorStore"):
    os.makedirs("../VectorStore")
    storage_context = StorageContext.from_defaults()
    storage_context.docstore.add_documents(nodes)

    automerging_index = VectorStoreIndex(
        leaf_nodes, storage_context=storage_context)

    automerging_index.storage_context.persist(persist_dir="../VectorStore")

else:
    automerging_index = load_index_from_storage(StorageContext.from_defaults(persist_dir="../VectorStore"))

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank
load_dotenv(find_dotenv())
cohere_rerank = CohereRerank(
    api_key=os.environ["COHERE_API_KEY"], 
    top_n=6,
)

In [ ]:
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

automerging_retriever = automerging_index.as_retriever(similarity_top_k=12)

retriever = AutoMergingRetriever(
    automerging_retriever,
    automerging_index.storage_context,
    verbose= True
)

auto_merging_engine = RetrieverQueryEngine.from_args(
    retriever, node_postprocessors=[cohere_rerank]
)

#### 4o-mini

In [ ]:
response = auto_merging_engine.query("suggest some bullets points I can use to begin teaching when I eventually find somebody to teach?")

In [ ]:
print(response)

### Evaluation

In [ ]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()


In [ ]:
import numpy as np
from trulens.apps.llamaindex import TruLlama
from trulens.core import Feedback
from trulens.providers.openai import OpenAI

# Initialize provider class
provider = OpenAI(model_engine="gpt-4o-mini")

# select context to be used in feedback. the location of context is app specific.

context = TruLlama.select_context(auto_merging_engine)

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(context.collect())  # collect context chunks into a list
    .on_output()
)

# Question/answer relevance between overall question and answer.
f_answer_relevance = Feedback(
    provider.relevance_with_cot_reasons, name="Answer Relevance"
).on_input_output()
# Question/statement relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(context)
    .aggregate(np.mean)
)

In [ ]:
tru_query_engine_recorder = TruLlama(
    auto_merging_engine,
    app_name="LlamaIndex_App",
    app_version="Advanced_Retriever",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [ ]:
import pandas as pd

# Read the CSV file and drop the 'Unnamed: 0' column
df = pd.read_csv("../dataset_eval/20_dataset.csv").drop(columns=['Unnamed: 0'])

# Extract the first 20 questions into a list
questions = df['question'][:20].tolist()

# Print the list of questions
print(questions)


In [ ]:
# evaluate
for question in questions:
    with tru_query_engine_recorder as recording:
        response = auto_merging_engine.query(question)

In [ ]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
# records.head()

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

In [ ]:
tru.get_leaderboard()

#### 3.5 turbo

In [ ]:
response = auto_merging_engine.query("what scriptures can help me teach about faith?")

In [ ]:
print(response)

In [ ]:
response = auto_merging_engine.query("suggest some bullets points I can use to begin teaching when I eventually find somebody to teach?")

In [ ]:
print(response)

In [ ]:
response = auto_merging_engine.query("How do i find people to teach?")


In [ ]:
print(response)